# Comprensión de los datos (EDA) · Avance #1 del PI (V1.0.1)

**Pregunta de negocio:** ¿qué características de un crédito y de su cliente se asocian con que el cliente **pague a tiempo**?

Estructura: 1) exploración inicial · 2) análisis univariable · 3) análisis bivariable · 4) análisis multivariable · 5) hallazgos y reglas de validación.

## 1. Exploración inicial

In [ ]:
# 1. Configuración y carga
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from IPython.display import display

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

RAIZ = Path.cwd()
while not (RAIZ / "mlops_pipeline").is_dir() and RAIZ != RAIZ.parent:
    RAIZ = RAIZ.parent
df = pd.read_csv(RAIZ / "Base_de_datos.csv")

TARGET = "Pago_atiempo"   # (por confirmar) 1 = pagó a tiempo · 0 = no pagó a tiempo
print(f"Filas: {df.shape[0]:,} | Columnas: {df.shape[1]}")

In [ ]:
# 2. Descripción general
df.info()
df.head()

In [ ]:
# 3. Caracterización: numéricas vs no numéricas y cardinalidad
num_cols = df.select_dtypes(include="number").columns.tolist()
cat_cols = df.select_dtypes(exclude="number").columns.tolist()
print("Numéricas   :", num_cols)
print("No numéricas:", cat_cols)
# 2 valores -> dicotómica | pocos (3 a 10) -> politómica | muchos -> continua, texto o ID
df.nunique().sort_values().to_frame("valores_distintos")

**Lectura:** `Pago_atiempo` y `tipo_laboral` son **dicotómicas**; `tipo_credito` es **politómica nominal** aunque venga como número (es un código, no una cantidad); `tendencia_ingresos` debería ser **ordinal** (Decreciente < Estable < Creciente). `fecha_prestamo` tiene casi un valor distinto por fila porque incluye la hora.

In [ ]:
# 4. Revisión de nulos
nulos = df.isna().sum()
resumen_nulos = pd.DataFrame({
    "nulos": nulos,
    "porcentaje": (nulos / len(df) * 100).round(2),
}).sort_values("porcentaje", ascending=False)
resumen_nulos[resumen_nulos["nulos"] > 0]

In [ ]:
# 5. Unificar la representación de nulos
VALORES_NULOS = ["", " ", "NA", "N/A", "na", "n/a", "NULL", "null",
                 "None", "none", "?", "-", "nan", "NaN"]
for col in df.columns:
    if df[col].dtype == "object" or pd.api.types.is_string_dtype(df[col]):
        df[col] = df[col].str.strip()
df = df.replace(VALORES_NULOS, np.nan)
print("Nulos totales después de unificar:", int(df.isna().sum().sum()))

In [ ]:
# 6. Eliminar variables irrelevantes (duplicados, constantes, IDs)
print("Filas duplicadas:", df.duplicated().sum())
df = df.drop_duplicates()
constantes = [c for c in df.columns if df[c].nunique(dropna=False) <= 1]
posibles_id = [c for c in df.columns if df[c].nunique() == len(df)]
print("Columnas constantes:", constantes)
print("Posibles IDs:", posibles_id)
# df = df.drop(columns=constantes + posibles_id)   # en esta base ambas listas salen vacías

In [ ]:
# 7. Tipos correctos
for col in df.columns:
    if df[col].dtype == "object" or pd.api.types.is_string_dtype(df[col]):
        numericos = int(pd.to_numeric(df[col], errors="coerce").notna().sum())
        if numericos:
            print(f"⚠️ {col}: {numericos} valores numéricos dentro de una columna de texto")

CATEGORIAS_TENDENCIA = ["Decreciente", "Estable", "Creciente"]
fuera = df["tendencia_ingresos"].notna() & ~df["tendencia_ingresos"].isin(CATEGORIAS_TENDENCIA)
print("tendencia_ingresos: valores fuera de catálogo pasados a nulo:", int(fuera.sum()))
df.loc[fuera, "tendencia_ingresos"] = np.nan

df["fecha_prestamo"] = pd.to_datetime(df["fecha_prestamo"], errors="coerce")
df["tipo_credito"] = df["tipo_credito"].astype("category")
df["tipo_laboral"] = df["tipo_laboral"].astype("category")
df["tendencia_ingresos"] = pd.Categorical(df["tendencia_ingresos"],
                                          categories=CATEGORIAS_TENDENCIA, ordered=True)
print("Rango de fechas:", df["fecha_prestamo"].min(), "→", df["fecha_prestamo"].max())

num_cols = [c for c in df.select_dtypes(include="number").columns if c != TARGET]
cat_cols = df.select_dtypes(include="category").columns.tolist()
df.dtypes

## 2. Análisis univariable

In [ ]:
# 8. describe() con los tipos ya ajustados
df[num_cols].describe().T

In [ ]:
# 8b. Estadística completa: tendencia central, dispersión y forma
def resumen_numerico(serie):
    q1, q3 = serie.quantile([0.25, 0.75])
    moda = serie.mode()
    return pd.Series({
        "media": serie.mean(), "mediana": serie.median(),
        "moda": moda.iloc[0] if not moda.empty else np.nan,
        "min": serie.min(), "max": serie.max(), "rango": serie.max() - serie.min(),
        "IQR": q3 - q1, "varianza": serie.var(), "desv_std": serie.std(),
        "skewness": serie.skew(), "kurtosis": serie.kurt(),
    })
df[num_cols].apply(resumen_numerico).T.round(3)

**Lectura:** `edad_cliente` llega a **123 años** (150 registros > 100), `salario_cliente` tiene un máximo de **22.000 millones** y 24 salarios en 0, `puntaje` tiene 135 valores negativos y `puntaje_datacredito` tiene 145 ceros (por eso su **moda** sale 0) y un valor negativo. Son errores de captura o centinelas, no clientes reales: anótalos como **reglas de validación**. Montos y salarios tienen **skewness** muy alta (cola larga a la derecha): su distribución se parece más a una **logarítmica** que a una gaussiana.

In [ ]:
# 9. Histograma + boxplot de las numéricas clave
COLS_UNI = ["capital_prestado", "plazo_meses", "edad_cliente",
            "salario_cliente", "puntaje", "puntaje_datacredito"]
for col in COLS_UNI:
    fig, ejes = plt.subplots(1, 2, figsize=(12, 3.5))
    sns.histplot(df[col].dropna(), kde=True, ax=ejes[0])
    sns.boxplot(x=df[col], ax=ejes[1])
    ejes[0].set_title(f"Distribución de {col}")
    ejes[1].set_title(f"Boxplot de {col}")
    plt.tight_layout()
    plt.show()

In [ ]:
# 10. Categóricas: value_counts + countplot
for col in cat_cols:
    print(f"\n=== {col} ===")
    print(df[col].value_counts(dropna=False).head(10))
    plt.figure(figsize=(6, 3))
    sns.countplot(y=df[col].astype(str), order=df[col].value_counts().index.astype(str)[:10])
    plt.title(col)
    plt.tight_layout()
    plt.show()

In [ ]:
# 11. Tabla pivote: monto y plazo por tipo laboral
pd.pivot_table(df, values=["capital_prestado", "plazo_meses"], index="tipo_laboral",
               aggfunc=["count", "mean", "median"], observed=True).round(0)

## 3. Análisis bivariable (respecto a la variable objetivo)

In [ ]:
# 12. Variable objetivo
assert TARGET in df.columns, "Define TARGET con el nombre exacto de la variable objetivo"
display(df[TARGET].value_counts(dropna=False))
display(df[TARGET].value_counts(normalize=True).round(4))

**Lectura:** ~95 % paga a tiempo y ~5 % no. Es una base **muy desbalanceada**: un modelo que diga "todos pagan" acierta el 95 % y no sirve para nada. Por eso en el Avance #2 mediremos con recall, F1 y ROC-AUC, no con accuracy.

In [ ]:
# 13. Numéricas vs objetivo
COLS_CLAVE = ["puntaje", "puntaje_datacredito", "huella_consulta",
              "plazo_meses", "edad_cliente", "capital_prestado"]
for col in COLS_CLAVE:
    plt.figure(figsize=(6, 3.5))
    sns.boxplot(x=df[TARGET].astype(str), y=df[col])
    plt.title(f"{col} según {TARGET}")
    plt.tight_layout()
    plt.show()
df.groupby(TARGET)[COLS_CLAVE].median().T

In [ ]:
# 14. Categóricas vs objetivo (proporción por fila)
for col in cat_cols:
    tabla = pd.crosstab(df[col], df[TARGET], normalize="index").round(3)
    tabla["n"] = df[col].value_counts()
    display(tabla)
    tabla.drop(columns="n").plot(kind="bar", stacked=True, figsize=(6, 3.5), title=f"{col} vs {TARGET}")
    plt.tight_layout()
    plt.show()

**Lectura:** la mediana de `puntaje` es ~95 en quienes pagan y ~25 en quienes no: separa casi perfecto. Antes de celebrar, **sospecha**: si ese puntaje se calcula con información posterior al crédito, sería **fuga de información** y no podría usarse para predecir. `huella_consulta` (más consultas → menos pago) y `puntaje_datacredito` muestran diferencias más moderadas. En `tipo_credito`, el código 6 tiene una tasa de pago mucho menor, pero con solo 21 créditos: ojo con sacar conclusiones de grupos pequeños.

## 4. Análisis multivariable

In [ ]:
# 15. Pairplot (muestra de 3.000 filas para que sea rápido)
COLS_PAIR = ["puntaje", "puntaje_datacredito", "capital_prestado", "cuota_pactada", "edad_cliente"]
df_pair = df[COLS_PAIR + [TARGET]].sample(n=3000, random_state=42)
df_pair[TARGET] = df_pair[TARGET].astype(str)
sns.pairplot(df_pair, hue=TARGET, corner=True, plot_kws={"alpha": 0.5})
plt.show()

In [ ]:
# 16. Matriz de correlación, pares más correlacionados y correlación con el objetivo
corr = df[num_cols + [TARGET]].corr()
plt.figure(figsize=(12, 9))
sns.heatmap(corr, annot=False, cmap="coolwarm", center=0)
plt.tight_layout()
plt.show()
pares = (corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1))
             .unstack().dropna().abs().sort_values(ascending=False))
display(pares.head(10).to_frame("|correlación|"))
corr[TARGET].drop(TARGET).sort_values().round(3).to_frame("correlación_con_objetivo")

In [ ]:
# 17. Tabla cruzada y dispersión con hue
display(pd.crosstab(df["tipo_laboral"], df["tendencia_ingresos"], margins=True))
plt.figure(figsize=(7, 4))
sns.scatterplot(data=df, x="capital_prestado", y="cuota_pactada", hue="tipo_laboral", alpha=0.5)
plt.title("Capital prestado vs cuota pactada")
plt.tight_layout()
plt.show()

## 5. Hallazgos, transformaciones y atributos derivados

| Hallazgo | Evidencia | Qué haremos en el Avance #2 |
|---|---|---|
|  |  |  |
|  |  |  |
|  |  |  |

**Transformaciones posibles**
- Imputar o marcar nulos de `promedio_ingresos_datacredito` y `tendencia_ingresos` (faltan juntos en ~27 % de las filas).
- Tratar valores imposibles (`edad_cliente` > 100, `salario_cliente` = 0, `puntaje_datacredito` = 0, puntajes negativos) como nulos o recortarlos.
- Escala logarítmica para montos con cola larga (`capital_prestado`, `salario_cliente`, `total_otros_prestamos`).
- One-hot encoding para `tipo_credito` y `tipo_laboral`; codificación ordinal para `tendencia_ingresos`.

**Atributos derivados que podrían servir**
- `cuota_pactada / salario_cliente` (carga de la cuota sobre el ingreso).
- `total_otros_prestamos / salario_cliente` (endeudamiento).
- Mes y día de la semana de `fecha_prestamo`.
- Indicador `tiene_mora = saldo_mora > 0`.
- Total de créditos por sector (financiero + cooperativo + real).

In [ ]:
# 19. Reglas de validación de datos para etapas posteriores
reglas = {                                   # cada máscara marca las filas que FALLAN la regla
    "sin filas duplicadas": df.duplicated(),
    "objetivo sin nulos": df[TARGET].isna(),
    "objetivo solo 0 o 1": ~df[TARGET].isin([0, 1]),
    "edad_cliente entre 18 y 100": ~df["edad_cliente"].between(18, 100),
    "salario_cliente mayor que 0": df["salario_cliente"] <= 0,
    "puntaje_datacredito mayor que 0": df["puntaje_datacredito"] <= 0,
    "saldo_principal <= saldo_total": df["saldo_principal"] > df["saldo_total"],
}
validacion = pd.DataFrame({"filas_que_fallan": {k: int(v.sum()) for k, v in reglas.items()}})
validacion["cumple"] = validacion["filas_que_fallan"] == 0
peor = df.isna().mean()
print(f"Columna con más nulos: {peor.idxmax()} ({peor.max():.1%}) → cumple la regla de máximo 30 %: {peor.max() <= 0.30}")
validacion